# Transaction Classification Benchmark

This notebook compares local Ollama transaction classification with batch sizes of 1, 10, and 20.

## Results

Batch sizes 10 and 20 produce the same classification quality, but batch size 20 is faster. Both batch sizes are significantly faster than processing one transaction per request. However, batch size 10 already shows unreliable output (frequently only 9 transactions get classified), therefore a batch size of 8-9 could give the most reliable outputs.

## 0. Imports

In [ ]:
# Step 0: Imports
import json
import sqlite3
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

## 1. Load vendor-enriched transactions

The benchmark classifies the same first 20 transactions for every batch size.

In [ ]:
database_path = Path("../database/vendor_mappings.sqlite")
category_path = Path("../categories.txt")

if not database_path.exists():
    raise ValueError(f"Vendor mappings database was not found: {database_path}")

categories = [
    line.strip()
    for line in category_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("#")
]

with sqlite3.connect(database_path) as connection:
    transactions = pd.read_sql_query("SELECT * FROM transactions", connection)

if "vendor" not in transactions.columns:
    raise ValueError("The transactions table has no vendor column. Run vendor extraction first.")

test_transactions = transactions.head(20).copy()
if test_transactions.empty:
    raise ValueError("No transactions were loaded from the database.")

## 2. Run benchmark

In [ ]:
OLLAMA_MODEL = "gemma3:4b"
OLLAMA_URL = "http://localhost:11434/api/generate"


def classify_vendors_batch(batch):
    vendor_list = "\n".join(
        f"{number}. {transaction['vendor']}"
        for number, (_, transaction) in enumerate(batch.iterrows(), start=1)
    )

    prompt = (
        "Classify each numbered transaction vendor into exactly one category. "
        "Use only the listed categories and choose Uncategorized when the vendor is unclear. "
        "Also return a confidence score from 0.00 to 1.00, with at most two decimal places, "
        "based on the vendor name alone. Return JSON only, for example: "
        '{\"categories\": {\"1\": {\"category\": \"Groceries\", \"confidence\": 0.95}}}.\n\n'
        + "Categories:\n- " + "\n- ".join(categories)
        + "\n\nVendors:\n" + vendor_list
    )

    try:
        response = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "format": "json", "stream": False},
            timeout=60,
        )
        response.raise_for_status()
        classified = json.loads(response.json()["response"])["categories"]
        result = [
            {
                "category": classified[str(number)]["category"].strip(),
                "confidence": round(float(classified[str(number)]["confidence"]), 2),
            }
            for number in range(1, len(batch) + 1)
        ]
    except (requests.RequestException, json.JSONDecodeError, KeyError, AttributeError, TypeError, ValueError) as error:
        raise ValueError(f"Could not classify vendors: {error}") from error

    if not {item["category"] for item in result}.issubset(categories):
        raise ValueError("Ollama returned a category outside categories.txt.")
    if not all(0 <= item["confidence"] <= 1 for item in result):
        raise ValueError("Ollama returned a confidence outside the 0-1 range.")

    return result


batch_sizes = [1, 10, 20]
performance_rows = []
review_table = test_transactions[["Beschreibung1", "Beschreibung2", "Beschreibung3", "vendor"]].copy()

for batch_size in batch_sizes:
    classifications = []
    request_count = 0
    start = time.perf_counter()

    for start_index in range(0, len(test_transactions), batch_size):
        batch = test_transactions.iloc[start_index : start_index + batch_size]
        classifications.extend(classify_vendors_batch(batch))
        request_count += 1

    performance_rows.append(
        {"Batch size": batch_size, "Requests": request_count, "Seconds": time.perf_counter() - start}
    )
    review_table[f"Category (batch {batch_size})"] = [
        item["category"] for item in classifications
    ]
    review_table[f"Confidence (batch {batch_size})"] = [
        item["confidence"] for item in classifications
    ]

performance = pd.DataFrame(performance_rows)
performance["Seconds per transaction"] = performance["Seconds"] / len(test_transactions)
performance["Speedup vs. batch size 1"] = (
    performance.loc[0, "Seconds per transaction"] / performance["Seconds per transaction"]
)

display(performance)
display(review_table)